In [0]:
from pyspark.sql import functions as f,Window
from pyspark.sql.functions import *
from pyspark.sql.types import *

customer_data = [
    # Valid Record
    (101, ' Rahul ', 'RAHUL@GMAIL.COM ', ' BANGALORE ', 'Gold', '98765-43210', '2025-01-02'),
    # Older version
    (101, ' Rahul ', 'rahul@gmail.com', 'Bangalore', 'Gold', '98765-43210', '2025-01-01'),
    # Exact duplicate
    (101, ' Rahul ', 'rahul@gmail.com', 'Bangalore', 'Gold', '98765-43210', '2025-01-01'),
    # Null Email
    (102, 'Amit', None, 'Pune', 'Silver', '98765-55210', '2025-01-05'),
    # Invalid Email
    (103, 'John', 'johngmail.com', 'Chennai', 'Gold', '98765-00210', '2025-01-04'),
    # Null Name
    (104, None, 'david@gmail.com', 'Mumbai', 'Bronze', '98765-00910', '2025-01-03'),
    # Latest Record
    (104, ' DAVID ', 'DAVID@GMAIL.COM ', ' MUMBAI ', 'Platinum', '98765-00910', '2025-01-10'),
    # Duplicate
    (104, ' DAVID ', 'DAVID@GMAIL.COM ', ' MUMBAI ', 'Platinum', '98765-00910', '2025-01-10'),
    # Null City
    (105, 'Sara', 'sara@gmail.com', None, 'Silver', '99865-88810', '2025-01-14'),
    # Null customer id
    (None, 'Ramesh', 'ramesh@gmail.com', 'Mysore', 'Gold', '98765-44410', '2025-01-05'),
    # Multiple Nulls
    (106, None, None, None, None, None, '2025-01-11'),
    # Future Date
    (107, 'Anu', 'anu@gmail.com', 'Kolkata', 'Gold', '98765-55510', '2030-01-06')
]

customer_schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("customer_name", StringType()),
    StructField("customer_email", StringType()),
    StructField("customer_city", StringType()),
    StructField("segment", StringType()),
    StructField("customer_phone", StringType()),
    StructField("load_dt", StringType())
])

orders_data = [
    (1001,101,"Laptop",50000,"2025-01-03"),
    (1002,102,"Mouse",1200,"2025-01-04"),
    (1003,103,"Keyboard",2500,"2025-01-05"),
    (1004,104,"Monitor",18000,"2025-01-11"),
    (1005,105,"Printer",15000,"2025-01-15"),
    # Customer doesn't exist
    (1006,108,"SSD",7000,"2025-01-05"),
    # Duplicate Order
    (1006,108,"SSD",7000,"2025-01-05"),
    # Invalid customer id
    (1007,None,"Tablet",25000,"2025-01-07"),
    # Negative Amount
    (1008,104,"Camera",-35000,"2025-01-06"),
    # Future Order
    (1009,101,"Mobile",65000,"2030-01-01"),
    # Dirty Product
    (1010,102," mouse ",1200,"2025-01-06"),
    # Customer not available in Dimension
    (1011,999,"TV",75000,"2025-01-06")
]

orders_schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("product", StringType()),
    StructField("amount", IntegerType()),
    StructField("order_dt", StringType())
])

customers = spark.createDataFrame(data=customer_data,schema=customer_schema)
orders = spark.createDataFrame(data=orders_data,schema=orders_schema)

customers = customers.dropna(subset=('customer_id'))
orders = orders.dropna(subset=('order_id'))

customers = customers.withColumn('load_dt',f.col('load_dt').cast(DateType()))
orders = orders.withColumn('order_dt',f.col('order_dt').cast(DateType()))

customers.show()
orders.show()

customers = customers.dropDuplicates()
orders = orders.dropDuplicates()

w = Window.partitionBy('customer_id').orderBy(f.col('load_dt').desc())
customers = customers.withColumn('rn',f.row_number().over(w)).filter(f.col('rn')==1).select('customer_id','customer_name','customer_email','customer_city','segment','customer_phone','load_dt')

customers = customers.fillna(
    {
        'customer_name':'Unknown',
        'customer_email':'Not Available',
        'customer_city':'Unknown City',
        'segment':'Unknown',
        'customer_phone':0
    }
)

customers = customers.withColumn('customer_name',f.initcap((f.trim(f.col('customer_name'))))).withColumn('customer_email',f.lower(f.trim(f.col('customer_email')))).withColumn('customer_city',f.initcap(f.trim(f.col('customer_city')))).withColumn('customer_phone',regexp_replace("customer_phone", "[^0-9]", ""))

email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
phone_regex = r'[^0-9]'

customers = customers.withColumn('data_quality',f.when(
    (f.col("customer_email").rlike(email_regex)) & (f.length(f.col('customer_phone')) == 10) &
    (f.col('load_dt')<=f.current_date())
,'Pass').otherwise('Fail'))


error_customers = customers.filter(f.col('data_quality') == 'Fail')
silver_customers = customers.filter(f.col('data_quality') == 'Pass')

orders = orders.dropDuplicates()
orders = orders.withColumn('product',f.initcap(f.trim(f.col('product'))))

orders = orders.withColumn('data_quality',f.when(
    (f.col('customer_id').isNotNull()) & (f.col('amount') >= 0) & (f.col('order_dt') <=f.current_date()),'Pass'
).otherwise('Fail'))

silver_orders = orders.filter(f.col('data_quality') == 'Pass')
error_orders = orders.filter(f.col('data_quality') == 'Fail')

silver_customers.show()
silver_orders.show()

silver_customers.write.format('delta').mode('overwrite').option('overwriteschema','true').saveAsTable('customers')
silver_orders.write.format('delta').mode('overwrite').option('overwriteschema','true').saveAsTable('orders')


